# Step 1: Setup and General Data Information

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = 'data/bankruptcy_italian_companies_2023.xlsx'
df = pd.read_excel(file_path)

# 1. General Data Information
print("Dataset Shape:", df.shape)
print("\nMissing Values per column (Top 5):")
print(df.isnull().sum().sort_values(ascending=False).head())

# Preview the target class distribution
print("\nInitial Target Class Distribution:")
print(df['Bankrupt'].value_counts())

Dataset Shape: (6819, 97)

Missing Values per column (Top 5):
Company                                                     0
Bankrupt                                                    0
 ROA(C) before interest and depreciation before interest    0
 ROA(A) before interest and % after tax                     0
 ROA(B) before interest and depreciation after tax          0
dtype: int64

Initial Target Class Distribution:
Bankrupt
0    6599
1     220
Name: count, dtype: int64


# Step 2: Data Cleaning and Target Label Mapping

In [3]:
# 2. Drop unique identifiers
if 'Company' in df.columns:
    df = df.drop(columns=['Company'])
    print("Dropped unique identifier column: 'Company'")

# Find and drop constant columns (columns where all values are exactly the same)
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
if constant_cols:
    df = df.drop(columns=constant_cols)
    print(f"Dropped constant columns: {constant_cols}")

# 3. Target Label Mapping
# Transform binary labels into categorical labels for interpretability
df['Bankrupt'] = df['Bankrupt'].map({0: 'Not Bankrupt', 1: 'Bankrupt'})

print("\nShape after cleaning:", df.shape)
print("Mapped Target Distribution:")
print(df['Bankrupt'].value_counts())

df.to_csv('data/df_cleaned_original.csv', index=False)

Dropped unique identifier column: 'Company'
Dropped constant columns: [' Net Income Flag']

Shape after cleaning: (6819, 95)
Mapped Target Distribution:
Bankrupt
Not Bankrupt    6599
Bankrupt         220
Name: count, dtype: int64


# Step 3: Feature Separation & Data Splitting

In [4]:
from sklearn.model_selection import train_test_split

# ==========================================
# EXPERIMENT TOGGLE: Change this value to test different splits!
# 0.20 = 80:20 split
# 0.30 = 70:30 split
TEST_SIZE = 0.20
# ==========================================

# Separate the features (X) and the target class (y)
X = df.drop(columns=['Bankrupt'])
y = df['Bankrupt']

# Split the data
# stratify=y ensures the bankrupt/not bankrupt ratio is maintained in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

print(f"Current Split: {int(100 - (TEST_SIZE*100))}:{int(TEST_SIZE*100)}")
print(f"Training set features shape: {X_train.shape}")
print(f"Testing set features shape: {X_test.shape}")

Current Split: 80:20
Training set features shape: (5455, 94)
Testing set features shape: (1364, 94)


# Step 4: Feature Selection via Information Gain (IG)

In [5]:
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from functools import partial

# Calculate Information Gain (Mutual Information) on the training set
ig_scores = mutual_info_classif(X_train, y_train, random_state=42)

# Visualize the top features
ig_df = pd.DataFrame({'Feature': X_train.columns, 'IG_Score': ig_scores})
ig_df = ig_df.sort_values(by='IG_Score', ascending=False)
print("Top 10 Features by Information Gain:")
print(ig_df.head(10))

# Select the top 20 features (adjust 'k' as needed for your experiments)
k_features = 20
ig_scorer = partial(mutual_info_classif, random_state=42)

selector = SelectKBest(score_func=ig_scorer, k=k_features)

# Apply the selection to both training and testing sets
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Get the names of the selected columns and convert back to DataFrames
selected_columns = X_train.columns[selector.get_support()]
X_train_sel_df = pd.DataFrame(X_train_selected, columns=selected_columns)
X_test_sel_df = pd.DataFrame(X_test_selected, columns=selected_columns)

print(f"\nSuccessfully isolated the top {k_features} predictive features.")
# Display all 20 selected features and their IG scores
print("All 20 Selected Features:")
display(ig_df.head(20))

Top 10 Features by Information Gain:
                                     Feature  IG_Score
39                      Borrowing dependency  0.045220
42     Net profit before tax/Paid-in capital  0.040991
18   Persistent EPS in the Last Four Seasons  0.040557
89        Net Income to Stockholder's Equity  0.040398
1     ROA(A) before interest and % after tax  0.037700
22           Per Share Net profit before tax  0.036989
16                   Net Value Per Share (A)  0.036707
36                              Debt ratio %  0.036246
93                       Equity to Liability  0.035844
90                       Liability to Equity  0.035763

Successfully isolated the top 20 predictive features.
All 20 Selected Features:


,Feature,IG_Score
39,Borrowing dependency,0.045220
42,Net profit before tax/Paid-in capital,0.040991
18,Persistent EPS in the Last Four Seasons,0.040557
89,Net Income to Stockholder's Equity,0.040398
1,ROA(A) before interest and % after tax,0.037700
22,Per Share Net profit before tax,0.036989
16,Net Value Per Share (A),0.036707
36,Debt ratio %,0.036246
93,Equity to Liability,0.035844
90,Liability to Equity,0.035763


# Step 5: Data Balancing via Synthetic Minority Over-Sampling Technique (SMOTE)

In [6]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE strictly to the pre-selected training data
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_sel_df, y_train)

print("Class distribution BEFORE SMOTE:")
print(y_train.value_counts())

print("\nClass distribution AFTER SMOTE:")
print(y_train_balanced.value_counts())

# Convert targets back to numeric (0 and 1) for XGBoost compatibility
y_train_balanced_numeric = y_train_balanced.map({'Not Bankrupt': 0, 'Bankrupt': 1})
y_test_numeric = y_test.map({'Not Bankrupt': 0, 'Bankrupt': 1})

# Dynamically set filenames based on the split
split_name = f"{int(100 - (TEST_SIZE*100))}{int(TEST_SIZE*100)}"

# Save the final preprocessed datasets
X_train_balanced.to_csv(f'data/X_train_processed_{split_name}.csv', index=False)
y_train_balanced_numeric.to_csv(f'data/y_train_processed_{split_name}.csv', index=False)
X_test_sel_df.to_csv(f'data/X_test_processed_{split_name}.csv', index=False)
y_test_numeric.to_csv(f'data/y_test_processed_{split_name}.csv', index=False)

print(f"\nPipeline complete! Preprocessed datasets for {split_name} split are saved and ready.")

#SAVE ALL DATA
# Create a specific folder in your actual Google Drive
import os
drive_path = '/content/drive/MyDrive/SHIELD_Project/data'
os.makedirs(drive_path, exist_ok=True)

# Dynamically set filenames based on the split
split_name = f"{int(100 - (TEST_SIZE*100))}{int(TEST_SIZE*100)}"

# Save the final preprocessed datasets DIRECTLY TO GOOGLE DRIVE
X_train_balanced.to_csv(f'{drive_path}/X_train_processed_{split_name}.csv', index=False)
y_train_balanced_numeric.to_csv(f'{drive_path}/y_train_processed_{split_name}.csv', index=False)
X_test_sel_df.to_csv(f'{drive_path}/X_test_processed_{split_name}.csv', index=False)
y_test_numeric.to_csv(f'{drive_path}/y_test_processed_{split_name}.csv', index=False)

print(f"\nSaved permanently to Google Drive at: {drive_path}")

Class distribution BEFORE SMOTE:
Bankrupt
Not Bankrupt    5279
Bankrupt         176
Name: count, dtype: int64

Class distribution AFTER SMOTE:
Bankrupt
Not Bankrupt    5279
Bankrupt        5279
Name: count, dtype: int64

Pipeline complete! Preprocessed datasets for 8020 split are saved and ready.

Saved permanently to Google Drive at: /content/drive/MyDrive/SHIELD_Project/data


# Step 6: Final Dataset Verification

In [7]:
import pandas as pd

# Define the split you just processed to load the correct files
# Change this to '7030' if you run the 70:30 split experiment
split_name = '8020'

# 1. Load the processed datasets
X_train_final = pd.read_csv(f'data/X_train_processed_{split_name}.csv')
y_train_final = pd.read_csv(f'data/y_train_processed_{split_name}.csv')
X_test_final = pd.read_csv(f'data/X_test_processed_{split_name}.csv')
y_test_final = pd.read_csv(f'data/y_test_processed_{split_name}.csv')

# 2. Print Shape Verification
print("=== FINAL DATASET OVERVIEW ===")
print(f"Training Features Shape: {X_train_final.shape} (Expect 20 columns)")
print(f"Testing Features Shape: {X_test_final.shape} (Expect 20 columns)")

# 3. Verify Class Distributions
print("\n=== CLASS DISTRIBUTION VERIFICATION ===")
print("Training Set (Should be perfectly 50/50 due to SMOTE):")
print(y_train_final.value_counts())

print("\nTesting Set (Should remain imbalanced/untouched):")
print(y_test_final.value_counts())

# 4. Quick Data Preview
print("\n=== FEATURE PREVIEW ===")
print("Top 5 rows of X_train:")
display(X_train_final.head())

=== FINAL DATASET OVERVIEW ===
Training Features Shape: (10558, 20) (Expect 20 columns)
Testing Features Shape: (1364, 20) (Expect 20 columns)

=== CLASS DISTRIBUTION VERIFICATION ===
Training Set (Should be perfectly 50/50 due to SMOTE):
Bankrupt
0           5279
1           5279
Name: count, dtype: int64

Testing Set (Should remain imbalanced/untouched):
Bankrupt
0           1320
1             44
Name: count, dtype: int64

=== FEATURE PREVIEW ===
Top 5 rows of X_train:


,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,Continuous interest rate (after tax),Net Value Per Share (B),Net Value Per Share (A),Net Value Per Share (C),Persistent EPS in the Last Four Seasons,Per Share Net profit before tax,Interest Expense Ratio,Debt ratio %,Net worth/Assets,Borrowing dependency,Net profit before tax/Paid-in capital,Retained Earnings to Total Assets,Total income/Total expense,Net Income to Total Assets,Net Income to Stockholder's Equity,Liability to Equity,Interest Coverage Ratio (Interest expense to EBIT),Equity to Liability
0,0.432896,0.501526,0.781172,0.171084,0.171084,0.171084,0.205068,0.163215,0.630295,0.097228,0.902772,0.374437,0.162235,0.925675,0.001967,0.778740,0.838851,0.278031,0.564693,0.039132
1,0.568956,0.614969,0.781790,0.183094,0.183094,0.183094,0.240238,0.196129,0.630653,0.069686,0.930314,0.370319,0.195144,0.948398,0.002579,0.844460,0.842990,0.276829,0.565206,0.055314
2,0.672598,0.695432,0.781964,0.234419,0.234419,0.234419,0.308500,0.271146,0.630638,0.084107,0.915893,0.370194,0.270289,0.973927,0.002913,0.883510,0.845723,0.277420,0.565186,0.045577
3,0.501243,0.557730,0.781679,0.175888,0.175888,0.175888,0.221046,0.175416,0.630671,0.048584,0.951416,0.370316,0.174399,0.937375,0.002293,0.811492,0.840905,0.276087,0.565242,0.079106
4,0.554526,0.614915,0.781749,0.195567,0.195567,0.195567,0.245155,0.198087,0.630620,0.085618,0.914382,0.369663,0.197006,0.945827,0.002528,0.839424,0.842882,0.277486,0.565168,0.044743
